# Candy Preference Survey Analysis

Clean and reshape the 2017 candy preference survey and examine demographic patterns.

**Tools:** Python, pandas, NumPy, Jupyter Notebook

In [1]:
import warnings
warnings.simplefilter(action='ignore', category=UserWarning)

import pandas as pd
import numpy as np

# Load the candy dataset
df_raw = pd.read_excel("data/Candy_Survey_2017.xlsx", engine="openpyxl")

# Display dataset information
print("Original dataset shape:", df_raw.shape)

# Show first 5 rows
df_raw.head()

Original dataset shape: (2460, 120)


,Internal ID,Q1: GOING OUT?,Q2: GENDER,Q3: AGE,Q4: COUNTRY,"Q5: STATE, PROVINCE, COUNTY, ETC",Q6 | 100 Grand Bar,Q6 | Anonymous brown globs that come in black and orange wrappers\t(a.k.a. Mary Janes),Q6 | Any full-sized candy bar,Q6 | Black Jacks,...,Q8: DESPAIR OTHER,Q9: OTHER COMMENTS,Q10: DRESS,Unnamed: 113,Q11: DAY,Q12: MEDIA [Daily Dish],Q12: MEDIA [Science],Q12: MEDIA [ESPN],Q12: MEDIA [Yahoo],"Click Coordinates (x, y)"
0,90258773,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,90272821,No,Male,44,USA,NM,MEH,DESPAIR,JOY,MEH,...,NaN,Bottom line is Twix is really the only candy w...,White and gold,NaN,Sunday,NaN,1.0,NaN,NaN,"(84, 25)"
2,90272829,NaN,Male,49,USA,Virginia,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,90272840,No,Male,40,us,or,MEH,DESPAIR,JOY,MEH,...,NaN,Raisins can go to hell,White and gold,NaN,Sunday,NaN,1.0,NaN,NaN,"(75, 23)"
4,90272841,No,Male,23,usa,exton pa,JOY,DESPAIR,JOY,DESPAIR,...,NaN,NaN,White and gold,NaN,Friday,NaN,1.0,NaN,NaN,"(70, 10)"


## Dataset Preparation

First thing we want to do before starting this assigment is to copy the data and find the main groups of columns. The survey has demographic questions, candy preference columns, comment columns, and some media and data related columns.

In [2]:
# Make a copy so the original imported data stays unchanged
df = df_raw.copy()

# Identify candy columns. In this dataset, candy ratings begin with 'Q6 |'
candy_cols = [col for col in df.columns if str(col).startswith('Q6 |')]

# Identify a few useful demographic columns
demographic_cols = [
    'Internal ID',
    'Q1: GOING OUT?',
    'Q2: GENDER',
    'Q3: AGE',
    'Q4: COUNTRY',
    'Q5: STATE, PROVINCE, COUNTY, ETC',
    'Q11: DAY'
]

print('Number of candy preference columns:', len(candy_cols))
print('Number of demographic columns selected:', len(demographic_cols))
print('Total columns in dataset:', df.shape[1])

Number of candy preference columns: 103
Number of demographic columns selected: 7
Total columns in dataset: 120


## 1. Chapter 7 - Filter Out Missing Data

In this method, I am removing rows that are missingt any important demographic fields. I went ahead and removed any rows that don't have values for gender, age, country, or survey day because those variables are used at a later point for grouping, pivoting, and data related tasks.

In [3]:
# Count rows before filtering
before_rows = len(df)

# Filter out rows missing important demographic information
required_cols = ['Q2: GENDER', 'Q3: AGE', 'Q4: COUNTRY', 'Q11: DAY']
df_filtered = df.dropna(subset=required_cols).copy()

after_rows = len(df_filtered)

print('Rows before filtering:', before_rows)
print('Rows after filtering:', after_rows)
print('Rows removed:', before_rows - after_rows)

df_filtered[required_cols].head()

Rows before filtering: 2460
Rows after filtering: 1700
Rows removed: 760


,Q2: GENDER,Q3: AGE,Q4: COUNTRY,Q11: DAY
1,Male,44,USA,Sunday
3,Male,40,us,Sunday
4,Male,23,usa,Friday
7,Male,33,canada,Friday
8,Male,40,Canada,Sunday


## 2. Chapter 7 - Fill In Missing Data

For the second method, I am filling in any missing values instead of removing them. For categorical columns, missing values were replaces with labels like Unknown or No Response. For the age column the values are converted to numeric format and missing ages are filled with the median age. These steps were down in order to perserve more of the rows for analysis.

In [4]:
# Create another cleaned copy from the filtered data
df_clean = df_filtered.copy()

# Convert age to numeric. Invalid entries become NaN.
df_clean['Q3: AGE'] = pd.to_numeric(df_clean['Q3: AGE'], errors='coerce')

# Fill missing age values with the median age
median_age = df_clean['Q3: AGE'].median()
df_clean['Q3: AGE'] = df_clean['Q3: AGE'].fillna(median_age)

# Fill missing categorical demographic fields
fill_values = {
    'Q1: GOING OUT?': 'Unknown',
    'Q2: GENDER': 'Unknown',
    'Q4: COUNTRY': 'Unknown',
    'Q5: STATE, PROVINCE, COUNTY, ETC': 'Unknown',
    'Q11: DAY': 'Unknown'
}
df_clean = df_clean.fillna(value=fill_values)

# Fill unanswered candy ratings with No Response
for col in candy_cols:
    df_clean[col] = df_clean[col].fillna('NO_RESPONSE')

print('Median age used to fill missing age values:', median_age)
print('Remaining missing values in selected demographic columns:')
print(df_clean[required_cols].isna().sum())

df_clean[required_cols].head()

Median age used to fill missing age values: 42.0
Remaining missing values in selected demographic columns:
Q2: GENDER     0
Q3: AGE        0
Q4: COUNTRY    0
Q11: DAY       0
dtype: int64


,Q2: GENDER,Q3: AGE,Q4: COUNTRY,Q11: DAY
1,Male,44.0,USA,Sunday
3,Male,40.0,us,Sunday
4,Male,23.0,usa,Friday
7,Male,33.0,canada,Friday
8,Male,40.0,Canada,Sunday


## 3. Chapter 8 - Combine and Merge Datasets

In this third method I am creating two related datasets from the original data and merging them together. The first dataset is restructured into a longer format where each of the rows will represent one rating for one candy. The second dataset is a candy lookup table that will have each candy name and a simple category. Lastly, we merged these two datasets using the candy name as the relationship key.

In [5]:
# Create a respondent demographic dataset
respondents = df_clean[demographic_cols].copy()

# Reshape candy columns into a long dataset
candy_long = df_clean.melt(
    id_vars=['Internal ID', 'Q2: GENDER', 'Q3: AGE', 'Q4: COUNTRY', 'Q11: DAY'],
    value_vars=candy_cols,
    var_name='Candy Column',
    value_name='Rating'
)

# Clean the candy name by removing the question prefix
candy_long['Candy'] = candy_long['Candy Column'].str.replace('Q6 | ', '', regex=False)

# Create a second related dataset: candy lookup table
def candy_category(name):
    name_lower = str(name).lower()
    if any(word in name_lower for word in ['chocolate', 'hershey', 'kit kat', 'snickers', 'twix', 'm&m', 'reese', 'milky way', 'butterfinger', 'toblerone', 'lindt']):
        return 'Chocolate'
    elif any(word in name_lower for word in ['fruit', 'peach', 'fish', 'skittles', 'starburst', 'sourpatch']):
        return 'Fruit / Gummy'
    elif any(word in name_lower for word in ['gum', 'tic tac', 'chiclets']):
        return 'Gum / Mint'
    elif any(word in name_lower for word in ['kale', 'healthy', 'whole wheat', 'white bread']):
        return 'Healthy / Non-Candy'
    elif any(word in name_lower for word in ['cash', 'vicodin', 'glow stick', 'pencils', 'comics']):
        return 'Non-Candy Item'
    else:
        return 'Other Candy'

candy_lookup = pd.DataFrame({'Candy': candy_long['Candy'].unique()})
candy_lookup['Candy Category'] = candy_lookup['Candy'].apply(candy_category)

# Merge the long candy response data with the lookup table
candy_merged = candy_long.merge(candy_lookup, on='Candy', how='left')

print('Long candy dataset shape:', candy_long.shape)
print('Candy lookup dataset shape:', candy_lookup.shape)
print('Merged dataset shape:', candy_merged.shape)

candy_merged.head()

Long candy dataset shape: (175100, 8)
Candy lookup dataset shape: (103, 2)
Merged dataset shape: (175100, 9)


,Internal ID,Q2: GENDER,Q3: AGE,Q4: COUNTRY,Q11: DAY,Candy Column,Rating,Candy,Candy Category
0,90272821,Male,44.0,USA,Sunday,Q6 | 100 Grand Bar,MEH,100 Grand Bar,Other Candy
1,90272840,Male,40.0,us,Sunday,Q6 | 100 Grand Bar,MEH,100 Grand Bar,Other Candy
2,90272841,Male,23.0,usa,Friday,Q6 | 100 Grand Bar,JOY,100 Grand Bar,Other Candy
3,90272854,Male,33.0,canada,Friday,Q6 | 100 Grand Bar,JOY,100 Grand Bar,Other Candy
4,90272858,Male,40.0,Canada,Sunday,Q6 | 100 Grand Bar,JOY,100 Grand Bar,Other Candy


## 4. Chapter 8 - Pivot the Data

In this method, we pivot the merged candy data to compare each of the average joy rates by gender and candy category. I first created a numeric indicator column for a rating. Then I used a pivot table to summarize the average joy rate.

In [6]:
# Create numeric indicators for candy ratings
candy_merged['Is Joy'] = (candy_merged['Rating'] == 'JOY').astype(int)
candy_merged['Is Meh'] = (candy_merged['Rating'] == 'MEH').astype(int)
candy_merged['Is Despair'] = (candy_merged['Rating'] == 'DESPAIR').astype(int)

# Pivot the data to compare joy rate by gender and candy category
joy_pivot = pd.pivot_table(
    candy_merged,
    values='Is Joy',
    index='Candy Category',
    columns='Q2: GENDER',
    aggfunc='mean'
).round(3)

joy_pivot

Q2: GENDER,Female,I'd rather not say,Male,Other
Candy Category,,,,
Chocolate,0.621,0.610,0.606,0.602
Fruit / Gummy,0.385,0.327,0.342,0.417
Gum / Mint,0.221,0.173,0.159,0.188
Healthy / Non-Candy,0.056,0.054,0.044,0.083
Non-Candy Item,0.303,0.276,0.297,0.354
Other Candy,0.306,0.304,0.302,0.298


## 5. Chapter 10 - Grouping with Functions

In this method I am grouping data using custom functions. The function will place each of the respondents into an age group and then the data is gruped in age groups to compare average candy joy, meh, and despair rates. This shows how a function will define the groups insteand of relying solely on existing columns.

In [7]:
# Define a grouping function for age values
def age_group(age):
    if age < 18:
        return 'Under 18'
    elif age < 30:
        return '18-29'
    elif age < 45:
        return '30-44'
    elif age < 60:
        return '45-59'
    else:
        return '60+'

# Apply the function to create an age group column
candy_merged['Age Group'] = candy_merged['Q3: AGE'].apply(age_group)

# Group by the function-created age group and calculate average rating indicators
age_group_summary = candy_merged.groupby('Age Group')[['Is Joy', 'Is Meh', 'Is Despair']].mean().round(3)

age_group_summary

,Is Joy,Is Meh,Is Despair
Age Group,,,
18-29,0.376,0.275,0.274
30-44,0.349,0.289,0.312
45-59,0.344,0.290,0.325
60+,0.308,0.284,0.347
Under 18,0.416,0.246,0.279


## 6. Chapter 10 - Cross Tabs

In this method I used a crosstab to compare two of the categorical variables which were gender and whether the respondent planned to go out. Cross tabs are useful to show the counts across each of the categories. I also included row and column totals using 'margins=True'

In [8]:
# Create a crosstab of gender by going-out response
gender_going_out = pd.crosstab(
    df_clean['Q2: GENDER'],
    df_clean['Q1: GOING OUT?'],
    margins=True
)

gender_going_out

Q1: GOING OUT?,No,Unknown,Yes,All
Q2: GENDER,,,,
Female,500,9,78,587
I'd rather not say,39,3,7,49
Male,857,43,140,1040
Other,20,1,3,24
All,1416,56,228,1700


## 7. Chapter 11 - Convert Between String and DateTime

In this method, I am converting string values into datetime values. The dataset included columns with values for specific days. The dataset didn't include a full calendar date so I wanted to create a survey date string from the day value and then convert it over into a datetime column.

In [9]:
# Map survey day strings to assumed survey dates around Halloween 2017
# Sunday is mapped to 2017-10-29 and Monday is mapped to 2017-10-30.
day_to_date = {
    'Sunday': '2017-10-29',
    'Monday': '2017-10-30'
}

df_clean['Survey Date String'] = df_clean['Q11: DAY'].map(day_to_date).fillna('2017-10-31')
df_clean['Survey Date'] = pd.to_datetime(df_clean['Survey Date String'])

print(df_clean[['Q11: DAY', 'Survey Date String', 'Survey Date']].head())
print()
print('Survey Date data type:', df_clean['Survey Date'].dtype)

  Q11: DAY Survey Date String Survey Date
1   Sunday         2017-10-29  2017-10-29
3   Sunday         2017-10-29  2017-10-29
4   Friday         2017-10-31  2017-10-31
7   Friday         2017-10-31  2017-10-31
8   Sunday         2017-10-29  2017-10-29

Survey Date data type: datetime64[ns]


/var/folders/x9/b8hlcxrx52vfv6cz4c_jrzcr0000gn/T/ipykernel_34975/387719207.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_clean['Survey Date String'] = df_clean['Q11: DAY'].map(day_to_date).fillna('2017-10-31')
/var/folders/x9/b8hlcxrx52vfv6cz4c_jrzcr0000gn/T/ipykernel_34975/387719207.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_clean['Survey Date'] = pd.to_datetime(df_clean['Survey Date String'])


## 8. Chapter 11 - Generate Date Range, Frequencies, and Date Offsets

Lastly, in this method I wanted to generate a date range and uses data offsets. I went ahead and created a daily date range for the survey window, count response, and include a follow up date that is seven days after the survey date. This will show time series style cleaning.

In [10]:
# Generate a daily date range covering the survey period
survey_range = pd.date_range(start=df_clean['Survey Date'].min(), end=df_clean['Survey Date'].max(), freq='D')
print('Generated survey date range:')
print(survey_range)

# Count responses by date and reindex to include every date in the generated range
responses_by_date = df_clean.groupby('Survey Date').size().reindex(survey_range, fill_value=0)
responses_by_date = responses_by_date.rename('Response Count').to_frame()
responses_by_date.index.name = 'Survey Date'

# Add a date offset: follow-up date is one week after the survey date
df_clean['Follow Up Date'] = df_clean['Survey Date'] + pd.DateOffset(days=7)

print()
print('Responses by survey date:')
display(responses_by_date)

print()
print('Example follow-up dates:')
df_clean[['Survey Date', 'Follow Up Date']].head()

Generated survey date range:
DatetimeIndex(['2017-10-29', '2017-10-30', '2017-10-31'], dtype='datetime64[ns]', freq='D')

Responses by survey date:


/var/folders/x9/b8hlcxrx52vfv6cz4c_jrzcr0000gn/T/ipykernel_34975/1627952145.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_clean['Follow Up Date'] = df_clean['Survey Date'] + pd.DateOffset(days=7)


,Response Count
Survey Date,
2017-10-29,630
2017-10-30,0
2017-10-31,1070



Example follow-up dates:


,Survey Date,Follow Up Date
1,2017-10-29,2017-11-05
3,2017-10-29,2017-11-05
4,2017-10-31,2017-11-07
7,2017-10-31,2017-11-07
8,2017-10-29,2017-11-05


## Final Summary

In this weeks assignment, I was able to select two methods from each chapter and include it here from one of the datasets provided. After using each of these methods I was able to show how the candy survey data can be cleaned, transformed, and summarized for analysis to gain more insight to what the data is telling us.